# Cross-beta Analysis — Aβ42 Variants

Analysis of Cross-beta results.

| Plot | What it shows |
|------|----------------|
| **Heatmap (absolute)** | Absolute amyloid score for each residue of each variant |
| **Heatmap (Δ vs WT)** | Deviation from Wildtype — where mutations alter amyloidogenicity |
| **Lollipop ΔAvg** | Average score across the entire sequence, vs WT |
| **Profiles** | Detailed line profiles of the top 5 destabilizing and stabilizing mutants |

In [ ]:
import pandas as pd
import ast
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
WT_COLOR = "#4ff7c0"
C_DESTAB = "#3b82f6"
C_NEUTRAL = "#64748b"
C_STAB = "#ef4444"
THRESH = 0.005

# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv("../predictions/cross-beta/cross-beta_result.csv", sep=";")
df["short"] = df["Query_name"].str.replace(r"_Abeta_?42$", "", regex=True)


def parse_scores(s):
    return [list(d.values())[0] for d in ast.literal_eval(s)]


df["scores"] = df["Amino_acids_score"].apply(parse_scores)
mat = np.array(df["scores"].tolist())  # (66, 42)

wt_idx = df[df["Query_name"] == "Wildtype_Abeta_42"].index[0]
wt_avg = df.loc[wt_idx, "Average_protein_prediction"]
wt_scores = mat[wt_idx]
wt_aa = [
    list(d.keys())[0] for d in ast.literal_eval(df.loc[wt_idx, "Amino_acids_score"])
]

df["Δavg"] = df["Average_protein_prediction"] - wt_avg
delta_mat = mat - wt_scores

# Sort by Δavg (stabilizing on top, destabilizing on bottom)
order = df.sort_values("Δavg", ascending=False).index
df_s = df.loc[order].reset_index(drop=True)
mat_s = mat[order]
delta_s = delta_mat[order]
wt_pos = df_s[df_s["Query_name"] == "Wildtype_Abeta_42"].index[0]

positions = np.arange(1, 43)
ticks_x = list(range(0, 42, 4)) + [41]


def score_color(d):
    if d < -THRESH:
        return C_DESTAB
    if d > THRESH:
        return C_STAB
    return C_NEUTRAL


print(f"Loaded variants: {len(df)}")
print(f"WT Average score: {wt_avg:.4f}")
print(f"Score range: {mat.min():.4f} … {mat.max():.4f}")
print("\nTop 5 destabilizing (smallest Δavg):")
print(
    df.nsmallest(5, "Δavg")[["short", "Average_protein_prediction", "Δavg"]].to_string(
        index=False
    )
)
print("\nTop 5 stabilizing (largest Δavg):")
print(
    df.nlargest(5, "Δavg")[["short", "Average_protein_prediction", "Δavg"]].to_string(
        index=False
    )
)

In [ ]:
# Plot 1 - Heatmap: absolute per-residue scores

fig, ax = plt.subplots(figsize=(16, 8), facecolor=BG)
ax.set_facecolor(BG)

im = ax.imshow(
    mat_s,
    aspect="auto",
    cmap=plt.cm.plasma,
    vmin=mat.min(),
    vmax=mat.max(),
    interpolation="nearest",
    origin="upper",
)

ax.set_xticks(ticks_x)
ax.set_xticklabels(
    [f"{t + 1}\n{wt_aa[t]}" for t in ticks_x],
    fontsize=7,
    fontfamily="monospace",
    color=MUTED,
)
ax.set_yticks(range(len(df_s)))
ax.set_yticklabels(df_s["short"].values, fontsize=5.2, fontfamily="monospace")
for tick, (_, row) in zip(ax.get_yticklabels(), df_s.iterrows()):
    tick.set_color(score_color(row["Δavg"]))
    tick.set_alpha(0.9)
ax.tick_params(axis="both", length=0, pad=3)

ax.axhline(wt_pos, color=WT_COLOR, lw=1.2, alpha=0.7, linestyle="--")
ax.text(
    42.4,
    wt_pos,
    "WT",
    va="center",
    color=WT_COLOR,
    fontsize=7,
    fontfamily="monospace",
    fontweight="bold",
)
for sp in ax.spines.values():
    sp.set_visible(False)

cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.01)
cbar.set_label(
    "Amyloid score per residue", color=MUTED, fontsize=8, fontfamily="monospace"
)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=7)
cbar.outline.set_edgecolor("#1e2540")

ax.set_xlabel(
    "Sequence position  (position / WT residue)",
    color=MUTED,
    fontsize=9,
    fontfamily="monospace",
    labelpad=8,
)
ax.set_title(
    "Cross-Beta  ·  Per-residue Amyloid Score  ·  All Aβ42 Variants",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    pad=12,
)

plt.tight_layout()
plt.savefig(
    "cross-beta_heatmap_abs.png",
    dpi=200,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to cross-beta_heatmap_abs.png")
plt.show()

In [ ]:
# Plot 2 — Heatmap: Δ per-residue scores (vs WT)
# Blue = lower than WT, Red = higher than WT

vlim = max(abs(delta_s).max(), 0.05)
norm_div = mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)

fig, ax = plt.subplots(figsize=(16, 8), facecolor=BG)
ax.set_facecolor(BG)

im2 = ax.imshow(
    delta_s,
    aspect="auto",
    cmap=plt.cm.RdBu_r,
    norm=norm_div,
    interpolation="nearest",
    origin="upper",
)

ax.set_xticks(ticks_x)
ax.set_xticklabels(
    [f"{t + 1}\n{wt_aa[t]}" for t in ticks_x],
    fontsize=7,
    fontfamily="monospace",
    color=MUTED,
)
ax.set_yticks(range(len(df_s)))
ax.set_yticklabels(df_s["short"].values, fontsize=5.2, fontfamily="monospace")
for tick, (_, row) in zip(ax.get_yticklabels(), df_s.iterrows()):
    tick.set_color(score_color(row["Δavg"]))
    tick.set_alpha(0.9)
ax.tick_params(axis="both", length=0, pad=3)

ax.axhline(wt_pos, color=WT_COLOR, lw=1.2, alpha=0.7, linestyle="--")
ax.text(
    42.4,
    wt_pos,
    "WT",
    va="center",
    color=WT_COLOR,
    fontsize=7,
    fontfamily="monospace",
    fontweight="bold",
)
for sp in ax.spines.values():
    sp.set_visible(False)

cbar2 = plt.colorbar(im2, ax=ax, shrink=0.7, pad=0.01)
cbar2.set_label(
    "Δ score  (mutant − WT)\nBlue = lower amyloid",
    color=MUTED,
    fontsize=8,
    fontfamily="monospace",
)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=7)
cbar2.outline.set_edgecolor("#1e2540")

ax.set_xlabel(
    "Sequence position  (position / WT residue)",
    color=MUTED,
    fontsize=9,
    fontfamily="monospace",
    labelpad=8,
)
ax.set_title(
    "Cross-Beta  ·  Δ Per-residue Score vs Wildtype  ·  All Aβ42 Variants",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    pad=12,
)

plt.tight_layout()
plt.savefig(
    "cross-beta_heatmap_delta.png",
    dpi=200,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to cross-beta_heatmap_delta.png")
plt.show()

In [ ]:
# Plot 3 — Lollipop: Δ Average Score vs WT (compact)

n = len(df_s)
fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG)
ax.set_facecolor(SURFACE)

for i in range(n):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.018, zorder=0)

ax.axvline(0, color=WT_COLOR, lw=0.9, linestyle="--", alpha=0.55)
ax.text(
    0,
    n + 0.1,
    "WT",
    ha="center",
    va="bottom",
    color=WT_COLOR,
    fontsize=7,
    fontfamily="monospace",
    fontweight="bold",
)

ax.axvspan(-0.12, -THRESH, color=C_DESTAB, alpha=0.055)
ax.axvspan(THRESH, 0.08, color=C_STAB, alpha=0.04)

for i, (_, row) in enumerate(df_s.iterrows()):
    c = score_color(row["Δavg"])
    ax.plot([0, row["Δavg"]], [i, i], color=c, lw=0.7, alpha=0.45)
    ax.scatter(row["Δavg"], i, color=c, s=18, zorder=3, linewidths=0)

fs = max(4.0, min(6.5, 8 * 20 / n))
ax.set_yticks(range(n))
ax.set_yticklabels(df_s["short"].values, fontsize=fs, fontfamily="monospace")
for tick, (_, row) in zip(ax.get_yticklabels(), df_s.iterrows()):
    tick.set_color(score_color(row["Δavg"]))
    tick.set_alpha(0.88)

ax.tick_params(axis="y", length=0, pad=3)
ax.tick_params(axis="x", colors=MUTED, labelsize=7)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xlim(-0.12, 0.08)
ax.set_ylim(-1, n)
ax.set_xlabel(
    "Δ Average amyloid score  (mutant − WT)\n"
    "< 0  →  lower amyloidogenicity  →  destabilizing mutation",
    color=MUTED,
    fontsize=8,
    fontfamily="monospace",
    labelpad=8,
)

n_d = (df_s["Δavg"] < -THRESH).sum()
n_n = (df_s["Δavg"].abs() <= THRESH).sum()
n_s = (df_s["Δavg"] > THRESH).sum()
lp = [
    mpatches.Patch(facecolor=C_DESTAB, label=f"Destabilizing (Δ<0) · n={n_d}"),
    mpatches.Patch(facecolor=C_NEUTRAL, label=f"Neutral · n={n_n}"),
    mpatches.Patch(facecolor=C_STAB, label=f"Stabilizing (Δ>0) · n={n_s}"),
]
ax.legend(
    handles=lp,
    loc="lower right",
    frameon=True,
    framealpha=0.15,
    edgecolor=MUTED,
    facecolor=SURFACE,
    fontsize=7,
    labelcolor=TEXT,
    handlelength=0.9,
)

ax.set_title(
    f"Cross-Beta  ·  Δ Average Score vs Wildtype\nWT = {wt_avg:.4f}",
    color="white",
    fontsize=10,
    fontfamily="monospace",
    fontweight="bold",
    pad=10,
)

plt.tight_layout()
plt.savefig(
    "cross-beta_lollipop.png",
    dpi=200,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to cross-beta_lollipop.png")
plt.show()

In [ ]:
# Plot 4 — Line profiles: WT + top 5 outliers

top_d = df.nsmallest(5, "Δavg")
top_s = df.nlargest(5, "Δavg")

fig, axes = plt.subplots(
    2, 1, figsize=(14, 10), facecolor=BG, gridspec_kw={"hspace": 0.45}
)

for ax_i, (subset, title_str, palette) in enumerate(
    [
        (
            top_d,
            "Top 5 destabilizing mutants  (most negative Δavg)",
            ["#60a5fa", "#a78bfa", "#f472b6", "#fb923c", "#34d399"],
        ),
        (
            top_s,
            "Top 5 stabilizing mutants  (most positive Δavg)",
            ["#f87171", "#fbbf24", "#a78bfa", "#60a5fa", "#34d399"],
        ),
    ]
):
    ax = axes[ax_i]
    ax.set_facecolor(SURFACE)

    ax.axhspan(0.5, 0.88, color="white", alpha=0.03)
    ax.axhline(0.5, color=MUTED, lw=0.6, linestyle=":", alpha=0.4)
    ax.text(
        42.5,
        0.5,
        "threshold\n0.5",
        va="center",
        color=MUTED,
        fontsize=6.5,
        fontfamily="monospace",
        alpha=0.7,
    )

    ax.plot(
        positions,
        wt_scores,
        color=WT_COLOR,
        lw=1.8,
        alpha=0.9,
        label="Wildtype",
        zorder=5,
    )
    ax.fill_between(positions, wt_scores, alpha=0.06, color=WT_COLOR)

    for (_, row), c in zip(subset.iterrows(), palette):
        label = f"{row['short']}  (Δ={row['Δavg']:+.4f})"
        ax.plot(positions, row["scores"], color=c, lw=1.1, alpha=0.82, label=label)

    ax.set_xlim(1, 42)
    ax.set_ylim(0.46, 0.88)
    ax.set_xticks(list(range(1, 43, 4)) + [42])
    ax.set_xticklabels(
        [f"{t}\n{wt_aa[t - 1]}" for t in list(range(1, 43, 4)) + [42]],
        fontsize=7,
        fontfamily="monospace",
        color=MUTED,
    )
    ax.tick_params(axis="x", length=0, pad=3)
    ax.tick_params(axis="y", colors=MUTED, labelsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor("#1e2540")

    ax.set_xlabel(
        "Sequence position  (position / WT residue)",
        color=MUTED,
        fontsize=8.5,
        fontfamily="monospace",
        labelpad=6,
    )
    ax.set_ylabel("Amyloid score", color=MUTED, fontsize=8.5, fontfamily="monospace")
    ax.legend(
        loc="upper left",
        frameon=True,
        framealpha=0.15,
        edgecolor=MUTED,
        facecolor=BG,
        fontsize=7.5,
        labelcolor=TEXT,
        handlelength=1.2,
    )
    ax.set_title(
        title_str,
        color="white",
        fontsize=10,
        fontfamily="monospace",
        fontweight="bold",
        pad=8,
    )

fig.suptitle(
    "Cross-Beta  ·  Per-residue Score Profile  ·  WT vs Outliers",
    color="white",
    fontsize=13,
    fontfamily="monospace",
    fontweight="bold",
    y=1.01,
)

plt.savefig(
    "cross-beta_profiles.png",
    dpi=200,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to cross-beta_profiles.png")
plt.show()